## <h3 align="center"> __Johns Hopkins University__</h3>
## <h3 align="center">__Whiting School of Engineering__</h3>
## <h3 align="center">__Engineering for Professionals__</h3>
## <h3 align="center">__685.801 Data Science: Independent Study__</h3>
## <h3 align="center">__Joint Embedding MoE Experiments__</h3>

---

This notebook mirrors the structure of `model_experiments.ipynb`, but it expands the experiment grid to compare: 

- standard MoE generalists built on `mobilenet_v3_small` and `convnext_small`
- joint-embedding generalists built on the same backbones
- both router families (`MLP`, `Layered`)
- all expert families (`MLP`, `Transformer`, `ResMLP`)

The joint-embedding variants add a supervised contrastive objective during continual-learning updates so the MoE compares baseline and JE backbones under the same task split.

---

## README (Execution & Setup)

- **Recommended runtime:** Google Colab with GPU or a local CUDA environment
- **Python version tested:** `3.12.x`
- **Primary packages:** `numpy`, `pandas`, `scikit-learn`, `matplotlib`, `seaborn`, `statsmodels`, `kagglehub`, `torch`, `torchvision`, `torchmetrics`
- **How to run:**
  1. Run cells in order.
  2. If using Colab, allow Drive mounting when prompted.
  3. Adjust the enabled model families or hyperparameters before the training runner cell if needed.
  4. Results are serialized into the configured results directory after each model finishes.

---

In [ ]:
%pip install numpy pandas scipy seaborn matplotlib scikit-learn statsmodels kagglehub torch torchvision torchao torchmetrics

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

REPO_URL = "https://github.com/ddimpfel/JHU_IS_26.git"
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

if IN_COLAB:
    drive.mount('/content/drive')
    repo_root = Path('/content') / REPO_NAME
    if not repo_root.exists():
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    CURRENT_WORK_DIR = repo_root
    os.chdir(CURRENT_WORK_DIR)
    RESULTS_DIR = Path('/content/drive/MyDrive/is26_models/joint_embedding_results')
else:
    CURRENT_WORK_DIR = Path.cwd()
    RESULTS_DIR = CURRENT_WORK_DIR / 'results' / 'joint_embedding'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if str(CURRENT_WORK_DIR) not in sys.path:
    sys.path.insert(0, str(CURRENT_WORK_DIR))

print(f'IN_COLAB={IN_COLAB}')
print(f'CURRENT_WORK_DIR={CURRENT_WORK_DIR}')
print(f'RESULTS_DIR={RESULTS_DIR}')

In [ ]:
import platform, numpy as np, pandas as pd, scipy, seaborn, matplotlib, sklearn, \
    statsmodels as sm, torch, torchvision, kagglehub, torchmetrics

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print(f'numpy:       {np.__version__}')
print(f'pandas:      {pd.__version__}')
print(f'scipy:       {scipy.__version__}')
print(f'sklearn:     {sklearn.__version__}')
print(f'seaborn:     {seaborn.__version__}')
print(f'matplotlib:  {matplotlib.__version__}')
print(f'statsmodels: {sm.__version__}')
print(f'torch:       {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'torchmetrics:{torchmetrics.__version__}')
print(f'kagglehub:   {kagglehub.__version__}')

In [ ]:
import random
from datetime import date
from itertools import product

from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.nn import CrossEntropyLoss, KLDivLoss
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset

from plots import plot_feature_vbars, visualize_images
from continual_learning import (
    CILComputerVisionModel,
    build_complete_dataloader,
    create_task_dataloaders,
    serialize_save_model_training,
    summarize_continual_metric_results,
)
from gre_model_base import (
    GeneralistRouterExperts,
    cost_proxy,
    expected_calibration_error,
    expert_metrics,
    router_entropy,
)
from joint_embedding import JointEmbeddingModule

## Config

In [ ]:
SEED = 7
NUM_WORKERS = 4 if IN_COLAB else 0
PIN_MEMORY = bool(IN_COLAB)
PERSIST_WORKERS = bool(IN_COLAB)
TOTAL_CLASSES = 9
BS_RESAMPLES = 1000

IMG_SIZE = 224
BATCH_SIZE = 64 if IN_COLAB else 8
EPOCHS = 8 if IN_COLAB else 5
NUM_TASKS = 3
EXEMPLAR_RATIO = 0.065
TASK_1_LR = 0.001
USE_CLASS_MASKING = True

NUM_EXPERTS = 6
HIDDEN_ROUTER_SIZE = 128
HIDDEN_EXPERT_SIZE = 128
DROPOUT = 0.1
TOP_K = 2
LAMBDA_AUX = 0.05

TFE_D_MODEL = 32
TFE_NHEAD = 4

JE_EMBEDDING_DIM = 256
JE_PROJECTION_DIM = 256
JE_FEATURE_KEY = 'projections'  # switch to 'embeddings' for an ablation
JE_TEMPERATURE = 0.07
JE_CONTRASTIVE_WEIGHT = 0.1

In [ ]:
np_rng = np.random.default_rng(seed=SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

cuda_available = torch.cuda.is_available()
device = 'cuda' if cuda_available else 'cpu'
tor_gen = torch.Generator(device='cpu').manual_seed(SEED)

print(f'CUDA available: {cuda_available}')
if cuda_available:
    current_device = torch.cuda.current_device()
    print(f'CUDA device count: {torch.cuda.device_count()}')
    print(f'Current device: {current_device}')
    print(f'Device name: {torch.cuda.get_device_name(current_device)}')
print(f'Using device: {device}')

## Data Loading

In [ ]:
GTSRB_DATASET_NAME = 'meowmeowmeowmeowmeow/gtsrb-german-traffic-sign'
csv_paths = {
    'train.csv': None,
    'test.csv': None,
    'meta.csv': None,
}

download_dir = Path(kagglehub.dataset_download(GTSRB_DATASET_NAME))
for csv in download_dir.rglob('*.csv'):
    if csv.name.lower() in csv_paths:
        csv_paths[csv.name.lower()] = csv

print('Kaggle dataset directory:', download_dir)
for name, path in csv_paths.items():
    print(f'{name}: {path}')

In [ ]:
from urllib.request import urlretrieve

signnames_url = 'https://raw.githubusercontent.com/georgesung/traffic_sign_classification_german/master/signnames.csv'
signnames_path = Path('signnames.csv')
if not signnames_path.exists():
    urlretrieve(signnames_url, signnames_path)

In [ ]:
train_csv = pd.read_csv(str(csv_paths['train.csv']))
train_df, val_df = train_test_split(
    train_csv,
    test_size=0.2,
    shuffle=True,
    stratify=train_csv.ClassId,
    random_state=SEED,
)
test_df = pd.read_csv(str(csv_paths['test.csv']))

prohibitory_sign_classes = [i for i in range(1, 11)]
prohibitory_sign_classes.remove(6)

prohib_train_df = train_df[train_df.ClassId.isin(prohibitory_sign_classes)].reset_index(drop=True)
prohib_val_df = val_df[val_df.ClassId.isin(prohibitory_sign_classes)].reset_index(drop=True)
prohib_test_df = test_df[test_df.ClassId.isin(prohibitory_sign_classes)].reset_index(drop=True)

plot_feature_vbars(
    prohib_train_df.ClassId,
    feature_name='Prohibitory Classes',
    title='Class Distribution of Prohibitory Signs (Training Data)',
)

psc = prohibitory_sign_classes.copy()
np_rng.shuffle(psc)
rand_prohib_classes = [psc[i:i + 3] for i in range(0, len(psc), 3)]

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array(prohibitory_sign_classes),
    y=prohib_train_df.ClassId.values,
)
class_weights_dict = dict(zip(prohibitory_sign_classes, weights))
weights_tensor = torch.tensor(list(class_weights_dict.values()), dtype=torch.float, device=device)

class_names = pd.read_csv('./signnames.csv')
meta_df = pd.read_csv(csv_paths['meta.csv'])
prohibitory_class_names = class_names[class_names.ClassId.isin(prohibitory_sign_classes)].reset_index(drop=True)
for task_classes in rand_prohib_classes:
    visualize_images(
        meta_df,
        download_dir,
        task_classes,
        prohibitory_class_names,
        'Random Tasks',
        nrows=1,
        ncols=3,
        figsize=(8, 4),
        fontsize_scaler=0.75,
    )

In [ ]:
class TrafficSignDataset(Dataset):
    def __init__(self, data_dir, image_file_paths, targets, transformer, class_mapping):
        if len(targets) != len(image_file_paths):
            raise AssertionError('Must have equal number of labels to images')

        self.data_dir = data_dir
        self.image_file_paths = list(image_file_paths)
        self.targets = list(targets)
        self.transformer = transformer
        self.class_mapping = class_mapping

    def __len__(self):
        return len(self.image_file_paths)

    def __getitem__(self, idx):
        img_path = os.path.join(self.data_dir, self.image_file_paths[idx])
        image = Image.open(img_path).convert('RGB')
        transformed_img = self.transformer(image)
        label = self.class_mapping[self.targets[idx]]
        target = {'label': torch.tensor(label, dtype=torch.long)}
        return transformed_img, target, str(img_path)

def collate_data(return_values):
    images = [item[0] for item in return_values]
    targets = [item[1] for item in return_values]
    paths = [item[2] for item in return_values]
    return images, targets, paths

transform_pipeline = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
class_mapping = {raw_id: i for i, raw_id in enumerate(sorted(prohibitory_sign_classes))}

train_dataset = TrafficSignDataset(
    download_dir,
    prohib_train_df.Path,
    prohib_train_df.ClassId,
    transform_pipeline,
    class_mapping,
)
val_dataset = TrafficSignDataset(
    download_dir,
    prohib_val_df.Path,
    prohib_val_df.ClassId,
    transform_pipeline,
    class_mapping,
)
test_dataset = TrafficSignDataset(
    download_dir,
    prohib_test_df.Path,
    prohib_test_df.ClassId,
    transform_pipeline,
    class_mapping,
)

train_dataloaders = create_task_dataloaders(
    train_dataset,
    rand_prohib_classes,
    batch_size=BATCH_SIZE,
    collate_fn=collate_data,
    generator=tor_gen,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persist_workers=PERSIST_WORKERS,
)
val_dataloaders = create_task_dataloaders(
    val_dataset,
    rand_prohib_classes,
    batch_size=BATCH_SIZE,
    collate_fn=collate_data,
    generator=tor_gen,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persist_workers=PERSIST_WORKERS,
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_data,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSIST_WORKERS,
    generator=tor_gen,
)

print(f'Train tasks: {len(train_dataloaders)}')
print(f'Validation tasks: {len(val_dataloaders)}')
print(f'Test examples: {len(test_dataset)}')

## Modeling Setup

In [ ]:
class MlpRouter(nn.Module):
    def __init__(self, input_size, num_experts):
        super().__init__()
        self.num_experts = num_experts
        self.gating = nn.Linear(input_size, num_experts, bias=False)

    def forward(self, x):
        logits = self.gating(x)
        if self.training:
            logits = logits + torch.randn_like(logits) * (1.0 / self.num_experts)
        return logits

class LayeredRouter(nn.Module):
    def __init__(self, input_size, num_experts):
        super().__init__()
        self.num_experts = num_experts
        self.ff1 = nn.Linear(input_size, input_size * 2)
        self.ff2 = nn.Linear(input_size * 2, num_experts, bias=False)

    def forward(self, x):
        logits = self.ff2(self.ff1(x))
        if self.training:
            logits = logits + torch.randn_like(logits) * (1.0 / self.num_experts)
        return logits

class MlpExpert(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout=0.1):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, output_size),
        )

    def forward(self, x):
        return self.network(x)

class TransformerExpert(nn.Module):
    def __init__(self, in_features, d_model, nhead, output_size):
        super().__init__()
        if in_features % d_model != 0:
            raise ValueError('d_model must perfectly divide in_features')
        self.num_patches = in_features // d_model
        self.d_model = d_model
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, d_model))
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True),
            num_layers=1,
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, output_size),
        )

    def forward(self, x):
        batch_size = x.shape[0]
        x = x.view(batch_size, self.num_patches, self.d_model)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) + self.pos_embed
        x = self.transformer(x)
        cls_output = x[:, 0, :]
        return self.classifier(cls_output)

class ResidualGatedMlpExpert(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_size),
            nn.Linear(input_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.block1 = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size * 2),
            nn.GLU(dim=1),
            nn.Dropout(dropout),
        )
        self.block2 = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size * 2),
            nn.GLU(dim=1),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.input_proj(x)
        x = x + self.block1(x)
        x = x + self.block2(x)
        return self.classifier(x)

In [ ]:
class JointEmbeddingRouterExperts(GeneralistRouterExperts):
    def __init__(self, *args, feature_key='projections', **kwargs):
        super().__init__(*args, **kwargs)
        self.feature_key = feature_key
        self._last_joint_embedding_outputs = None

    def reset_routing_state(self):
        super().reset_routing_state()
        self._last_joint_embedding_outputs = None

    def get_joint_embedding_loss(self, labels):
        if self._last_joint_embedding_outputs is None:
            param = next(self.parameters())
            return torch.zeros((), device=param.device, dtype=param.dtype)

        if self.feature_key == 'embeddings':
            return self.generalist.compute_contrastive_loss(
                labels=labels,
                embeddings=self._last_joint_embedding_outputs['embeddings'],
            )

        return self.generalist.compute_contrastive_loss(
            labels=labels,
            projections=self._last_joint_embedding_outputs['projections'],
        )

    def forward(self, x):
        encoded = self.generalist.backbone.encode(x)
        generalist_logits = encoded['logits']
        features = encoded[self.feature_key]
        if features.ndim != 2:
            features = torch.flatten(features, 1)

        router_logits = self.router(features)
        router_probs = F.softmax(router_logits, dim=1)
        topk_probs, topk_indices = torch.topk(router_probs, k=self.k, dim=1)
        topk_weights = topk_probs / topk_probs.sum(dim=1, keepdim=True).clamp_min(1e-8)

        experts_logits = torch.zeros_like(generalist_logits, device=features.device, dtype=features.dtype)
        for expert_idx, expert in enumerate(self.experts):
            selected_rows, selected_slots = torch.where(topk_indices == expert_idx)
            if selected_rows.numel() == 0:
                continue
            expert_features = features[selected_rows]
            expert_logits = expert(expert_features)
            expert_weights = topk_weights[selected_rows, selected_slots].unsqueeze(1)
            experts_logits[selected_rows] += expert_logits * expert_weights

        self._last_router_probs = router_probs.detach()
        self._last_topk_indices = topk_indices.detach()
        self._last_aux_loss = self._compute_auxiliary_loss(router_probs, topk_indices)
        self._last_joint_embedding_outputs = {
            key: value
            for key, value in encoded.items()
            if isinstance(value, torch.Tensor)
        }
        return (generalist_logits + experts_logits) / self.temperature

class JointEmbeddingCILComputerVisionModel(CILComputerVisionModel):
    def __init__(self, *args, lambda_contrastive=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.lambda_contrastive = lambda_contrastive

    def _get_joint_embedding_loss(self, target):
        if hasattr(self.model, 'get_joint_embedding_loss'):
            return self.lambda_contrastive * self.model.get_joint_embedding_loss(target)
        logits_dtype = next(self.model.parameters()).dtype
        return torch.zeros((), device=self.device, dtype=logits_dtype)

    def loss(self, loss_fn, images, target):
        new_pred = self.model(images)
        loss_pred = loss_fn(self._mask_logits(new_pred), target)
        aux_loss = self._get_model_aux_loss()
        contrastive_loss = self._get_joint_embedding_loss(target)

        if self.prev_model is None:
            return loss_pred + aux_loss + contrastive_loss

        with torch.no_grad():
            old_pred = self.prev_model(images)

        kd_idx = self.prev_seen_classes_tensor
        new_pred_kd = new_pred[:, kd_idx]
        old_pred_kd = old_pred[:, kd_idx]
        loss_kd = KLDivLoss(reduction='batchmean')(
            F.log_softmax(new_pred_kd / self.kd_temperature, dim=1),
            F.softmax(old_pred_kd / self.kd_temperature, dim=1),
        ) * (self.kd_temperature * self.kd_temperature)
        return (1 - self.lambda_kd) * loss_pred + self.lambda_kd * loss_kd + aux_loss + contrastive_loss

In [ ]:
JOINT_EMBEDDING_FACTORIES = {
    'JE MobileNet Small': lambda: JointEmbeddingModule(
        num_classes=TOTAL_CLASSES,
        backbone='mobilenet_v3_small',
        embedding_dim=JE_EMBEDDING_DIM,
        projection_dim=JE_PROJECTION_DIM,
        pretrained=True,
        temperature=JE_TEMPERATURE,
    ),
    'JE ConvNeXt Small': lambda: JointEmbeddingModule(
        num_classes=TOTAL_CLASSES,
        backbone='convnext_tiny',
        embedding_dim=JE_EMBEDDING_DIM,
        projection_dim=JE_PROJECTION_DIM,
        pretrained=True,
        temperature=JE_TEMPERATURE,
    ),
}

ROUTER_FACTORIES = {
    'MLP Router': MlpRouter,
    'Layered Router': LayeredRouter,
}

EXPERT_FACTORIES = {
    'MLP Experts': MlpExpert,
    'Transformer Experts': TransformerExpert,
    'ResMLP Experts': ResidualGatedMlpExpert,
}

def build_expert_factory(expert_name, generalist_out_dim):
    def expert_factory():
        if expert_name == 'MLP Experts':
            return MlpExpert(
                input_size=generalist_out_dim,
                hidden_size=HIDDEN_EXPERT_SIZE,
                output_size=TOTAL_CLASSES,
                dropout=DROPOUT,
            )
        if expert_name == 'Transformer Experts':
            if generalist_out_dim % TFE_D_MODEL != 0:
                raise ValueError(
                    f'TransformerExpert requires in_features divisible by d_model; got {generalist_out_dim} and {TFE_D_MODEL}.'
                )
            return TransformerExpert(
                in_features=generalist_out_dim,
                d_model=TFE_D_MODEL,
                nhead=TFE_NHEAD,
                output_size=TOTAL_CLASSES,
            )
        if expert_name == 'ResMLP Experts':
            return ResidualGatedMlpExpert(
                input_size=generalist_out_dim,
                hidden_size=HIDDEN_EXPERT_SIZE,
                output_size=TOTAL_CLASSES,
                dropout=DROPOUT,
            )
        raise KeyError(f'Unknown expert: {expert_name}')
    return expert_factory

def build_model_specs(k=TOP_K):
    specs = {}
    for generalist_name, router_name, expert_name in product(
        JOINT_EMBEDDING_FACTORIES.keys(),
        ROUTER_FACTORIES.keys(),
        EXPERT_FACTORIES.keys(),
    ):
        def model_factory(generalist_name=generalist_name, router_name=router_name, expert_name=expert_name):
            generalist = JOINT_EMBEDDING_FACTORIES[generalist_name]()
            generalist_out_dim = JE_PROJECTION_DIM if JE_FEATURE_KEY == 'projections' else JE_EMBEDDING_DIM
            router = ROUTER_FACTORIES[router_name](
                input_size=generalist_out_dim,
                num_experts=NUM_EXPERTS,
            )
            return JointEmbeddingRouterExperts(
                generalist=generalist,
                router=router,
                expert_factory=build_expert_factory(expert_name, generalist_out_dim),
                num_experts=NUM_EXPERTS,
                num_classes=TOTAL_CLASSES,
                k=k,
                lambda_aux=LAMBDA_AUX,
                feature_key=JE_FEATURE_KEY,
            )

        model_name = f'{generalist_name} + {router_name} + {expert_name}'
        specs[model_name] = {
            'family': 'Joint Embedding',
            'backbone': generalist_name,
            'feature_space': JE_FEATURE_KEY,
            'uses_joint_embedding': True,
            'factory': model_factory,
        }

    return specs

In [ ]:
def build_cil_model(model_factory, uses_joint_embedding, exemplar_ratio=EXEMPLAR_RATIO, use_class_masking=USE_CLASS_MASKING):
    cil_cls = JointEmbeddingCILComputerVisionModel if uses_joint_embedding else CILComputerVisionModel
    kwargs = {
        'model': model_factory(),
        'exemplar_ratio': exemplar_ratio,
        'rng': np_rng,
        'device': device,
        'use_class_masking': use_class_masking,
    }
    if uses_joint_embedding:
        kwargs['lambda_contrastive'] = JE_CONTRASTIVE_WEIGHT
    return cil_cls(**kwargs)

def collect_gre_post_training_metrics(cil_model, eval_dataloader, num_classes=TOTAL_CLASSES, device=device):
    model = cil_model.model
    if not isinstance(model, GeneralistRouterExperts):
        return {}

    entropy_metrics = router_entropy(model, eval_dataloader, device=device)
    expert_summary = expert_metrics(model, eval_dataloader, device=device)
    ece = expected_calibration_error(
        model,
        num_classes=num_classes,
        val_data=eval_dataloader,
        device=device,
    )
    params = model.get_model_parameters_by_component()
    expected_calls = expert_summary['Expected Expert Calls']

    return {
        'Post Training Router Entropy': entropy_metrics['Mean Router Entropy'],
        'Post Training Normalized Router Entropy': entropy_metrics['Normalized Router Entropy'],
        'Post Training Router Entropy Std': entropy_metrics['Std Router Entropy'],
        'Post Training Avg Router Prob': entropy_metrics['Avg Router Prob'],
        'Post Training ECE': ece,
        'Post Training Expected Expert Calls': expected_calls,
        'Post Training Expert Utilization Ratio': expert_summary['Expert Selection Ratio'],
        'Post Training Cost Proxy': cost_proxy(expected_calls, params),
    }

def train_and_evaluate_model(
    cil_model,
    optimizer_cls,
    loss_fn,
    train_dataloaders,
    val_dataloaders,
    epochs=5,
    num_tasks=3,
    base_lr=0.001,
    device='cpu',
    verbose=False,
):
    model = cil_model.model.to(device)
    baseline_accuracy = 0.0
    eval_matrices = {'Micro F1': np.zeros((num_tasks, num_tasks))}
    forward_eval_matrices = {'Micro F1': np.full((num_tasks, num_tasks), np.nan)}
    baseline_forward_scores = {'Micro F1': np.zeros(num_tasks)}
    task_training_times = []
    loss_history_per_task = []
    pred_cache = {}
    full_val_dataloader = build_complete_dataloader(val_dataloaders, shuffle=False)

    for t in range(num_tasks):
        _, baseline_metrics = cil_model.evaluate(
            val_dataloaders[t],
            loss_fn,
            apply_class_masking=False,
        )
        baseline_forward_scores['Micro F1'][t] = baseline_metrics['Micro F1']

    for i in range(num_tasks):
        print(f'\rStage {i + 1}/{num_tasks}: Model {model.__class__.__name__}', end='', flush=True)
        if i > 0:
            _, pre_task_metrics = cil_model.evaluate(
                val_dataloaders[i],
                loss_fn,
                apply_class_masking=False,
            )
            forward_eval_matrices['Micro F1'][i, i - 1] = pre_task_metrics['Micro F1']

        current_lr = base_lr if i == 0 else base_lr * 0.1
        optimizer = optimizer_cls(cil_model.model.parameters(), lr=current_lr)

        start_time = time.time()
        task_loss_history = cil_model.train(train_dataloaders[i], optimizer, loss_fn, epochs, verbose=verbose)
        compute_time = time.time() - start_time
        task_training_times.append(compute_time)
        loss_history_per_task.append(task_loss_history)

        for t in range(i + 1):
            _, val_metrics, task_logits, task_targets, task_paths = cil_model.evaluate(
                val_dataloaders[t],
                loss_fn,
                return_logits=True,
            )
            eval_matrices['Micro F1'][t, i] = val_metrics['Micro F1']
            pred_cache[(i, t)] = {
                'logits': task_logits,
                'targets': task_targets,
                'paths': task_paths,
                'metrics': val_metrics,
            }
            if i == 0:
                baseline_accuracy = val_metrics['Micro F1']

    print(f'\nTotal compute time = {np.sum(task_training_times):.2f}s')

    micro_results = summarize_continual_metric_results(
        eval_matrix=eval_matrices['Micro F1'],
        forward_eval_matrix=forward_eval_matrices['Micro F1'],
        baseline_forward_scores=baseline_forward_scores['Micro F1'],
        suffix='Micro F1',
    )
    full_val_loss, full_val_metrics = cil_model.evaluate(full_val_dataloader, loss_fn)

    results = {
        **micro_results,
        'AvgAcc': micro_results['AvgAcc Micro F1'],
        'Backward Transfer': micro_results['Backward Transfer Micro F1'],
        'Backward Transfer Per Task': micro_results['Backward Transfer Per Task Micro F1'],
        'Forward Transfer': micro_results['Forward Transfer Micro F1'],
        'Forward Transfer Per Task': micro_results['Forward Transfer Per Task Micro F1'],
        'Task Forgetting List': micro_results['Task Forgetting List Micro F1'],
        'Average Forgetting': micro_results['Average Forgetting Micro F1'],
        'Eval Matrix': micro_results['Eval Matrix Micro F1'],
        'Baseline Accuracy': baseline_accuracy,
        'Full Validation Loss': full_val_loss,
        'Full Validation F1': full_val_metrics['Micro F1'],
        'Full Validation Macro F1': full_val_metrics['Macro F1'],
        'Full Validation Micro F1': full_val_metrics['Micro F1'],
        'Full Validation Weighted F1': full_val_metrics['Weighted F1'],
        'Training Time per Stage': task_training_times,
        'Total Compute Time (s)': np.sum(task_training_times),
        'Num Parameters': sum(p.numel() for p in model.parameters()),
    }
    return results, loss_history_per_task, pred_cache

def run_model_comparisons(
    model_specs,
    train_dataloaders,
    val_dataloaders,
    optimizer_cls=AdamW,
    loss_fn=None,
    epochs=EPOCHS,
    num_tasks=NUM_TASKS,
    base_lr=TASK_1_LR,
    device=device,
    verbose=False,
):
    if loss_fn is None:
        loss_fn = CrossEntropyLoss(weight=weights_tensor)

    comparison_rows = []
    loss_histories = {}
    prediction_caches = {}
    trained_models = {}
    full_val_dataloader = build_complete_dataloader(val_dataloaders, shuffle=False)

    for model_name, spec in model_specs.items():
        print(f'\n=== Running {model_name} ===')
        cil_model = build_cil_model(spec['factory'], spec['uses_joint_embedding'])
        results, loss_history_per_task, pred_cache = train_and_evaluate_model(
            cil_model=cil_model,
            optimizer_cls=optimizer_cls,
            loss_fn=loss_fn,
            train_dataloaders=train_dataloaders,
            val_dataloaders=val_dataloaders,
            epochs=epochs,
            num_tasks=num_tasks,
            base_lr=base_lr,
            device=device,
            verbose=verbose,
        )
        results.update(collect_gre_post_training_metrics(cil_model, full_val_dataloader, device=device))
        comparison_rows.append({
            'Model': model_name,
            'Family': spec['family'],
            'Backbone': spec['backbone'],
            'Feature Space': spec['feature_space'],
            **results,
        })
        loss_histories[model_name] = loss_history_per_task
        prediction_caches[model_name] = pred_cache
        trained_models[model_name] = cil_model

        model_out_path = RESULTS_DIR / f'{model_name}_{date.today()}'
        serialize_save_model_training(results, pred_cache, f'{model_out_path}.json')
        cil_model.save(filename=f'{model_out_path}.pth')

    comparison_df = pd.DataFrame(comparison_rows)
    return comparison_df, trained_models, loss_histories, prediction_caches

## Model Training Pipeline

In [ ]:
model_specs = build_model_specs()
print(f'Model count: {len(model_specs)}')
list(model_specs.keys())[:5]

In [ ]:
comparison_df, trained_models, loss_histories, prediction_caches = run_model_comparisons(
    model_specs=model_specs,
    train_dataloaders=train_dataloaders,
    val_dataloaders=val_dataloaders,
    optimizer_cls=AdamW,
    loss_fn=CrossEntropyLoss(weight=weights_tensor),
    epochs=EPOCHS,
    num_tasks=NUM_TASKS,
    base_lr=TASK_1_LR,
    device=device,
    verbose=False,
)
comparison_df.head()

In [ ]:
metric_columns = [
    'Model',
    'Family',
    'Backbone',
    'Feature Space',
    'AvgAcc Micro F1',
    'Backward Transfer',
    'Forward Transfer',
    'Average Forgetting',
    'Full Validation Loss',
    'Full Validation Micro F1',
    'Baseline Accuracy',
    'Total Compute Time (s)',
]
available_metric_columns = [column for column in metric_columns if column in comparison_df.columns]
summary_df = comparison_df.loc[:, available_metric_columns].copy()
summary_df = summary_df.sort_values(by=['AvgAcc', 'Full Validation Micro F1'], ascending=False).reset_index(drop=True)
display(summary_df.style.format({
    'AvgAcc Micro F1': '{:.4f}',
    'Backward Transfer': '{:.4f}',
    'Forward Transfer': '{:.4f}',
    'Average Forgetting': '{:.4f}',
    'Full Validation Loss': '{:.4f}',
    'Full Validation Micro F1': '{:.4f}',
    'Baseline Accuracy': '{:.4f}',
    'Total Compute Time (s)': '{:.1f}',
}))

fig, axes = plt.subplots(2, 2, figsize=(20, 14), constrained_layout=True)
plot_metrics = ['AvgAcc Micro F1', 'Average Forgetting', 'Full Validation Micro F1']
for ax, metric in zip(axes.flat, plot_metrics):
    plot_df = summary_df.sort_values(metric, ascending=False)
    sns.barplot(data=plot_df, x=metric, y='Model', hue='Family', ax=ax)
    ax.set_title(metric)
    ax.set_xlabel(metric)
    ax.set_ylabel('')
    ax.legend(loc='best')
plt.show()
summary_df

In [ ]:
gre_metric_columns = [
    'Model',
    'Family',
    'Backbone',
    'Feature Space',
    'Post Training Router Entropy',
    'Post Training Normalized Router Entropy',
    'Post Training Router Entropy Std',
    'Post Training ECE',
    'Post Training Expected Expert Calls',
    'Post Training Cost Proxy',
    'Post Training Expert Utilization Ratio',
]
available_gre_columns = [column for column in gre_metric_columns if column in comparison_df.columns]
gre_metrics_df = comparison_df.loc[:, available_gre_columns].copy()
display(gre_metrics_df.style.format({
    'Post Training Router Entropy': '{:.4f}',
    'Post Training Normalized Router Entropy': '{:.4f}',
    'Post Training Router Entropy Std': '{:.4f}',
    'Post Training ECE': '{:.4f}',
    'Post Training Expected Expert Calls': '{:.4f}',
    'Post Training Cost Proxy': '{:.0f}',
}))
gre_metrics_df